In [43]:
!pip install -q \
torch \
transformers \
datasets \
accelerate \
scikit-learn \
pandas \
numpy \
huggingface_hub

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        PROJECT_ROOT 
    )
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_CLEAN_DIR = DATA_DIR / "clean"

SRC_DIR = PROJECT_ROOT / "src"
MODEL_DIR = SRC_DIR / "model"

REPORTS_DIR = PROJECT_ROOT / "reports"
TRANSFORMER_REPORTS_DIR = REPORTS_DIR / "transformers"
COMPARISON_REPORTS_DIR = REPORTS_DIR / "comparisons"

CLEAN_DATA_PATH = DATA_CLEAN_DIR / "mental_health_detection_clean.csv"

TRANSFORMER_RESULTS_PATH = TRANSFORMER_REPORTS_DIR / "transformer_test_results.csv"
TRANSFORMER_CLASS_RECALL_PATH = TRANSFORMER_REPORTS_DIR / "transformer_class_recall.csv"
TRANSFORMER_PREDICTIONS_PATH = TRANSFORMER_REPORTS_DIR / "transformer_test_predictions.csv"

for directory in [DATA_CLEAN_DIR, MODEL_DIR, TRANSFORMER_REPORTS_DIR, COMPARISON_REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✅ Transformer paths ready")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CLEAN_DATA_PATH:", CLEAN_DATA_PATH)

✅ Transformer paths ready
PROJECT_ROOT: /content/mental_health_final_project
CLEAN_DATA_PATH: /content/mental_health_final_project/data/clean/mental_health_detection_clean.csv


In [45]:
import os
import random
import shutil
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
# ============================================================
# TRANSFORMER IMPORTS
# ============================================================

import shutil
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, recall_score

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

In [46]:
RANDOM_STATE = 42

TEXT_COL = "body"
MASKED_COL = "body_masked"
TARGET_COL = "category"

USE_MASKED_TEXT = False

TRANSFORMER_MAX_LENGTH = 256
TRANSFORMER_BATCH_SIZE = 8
TRANSFORMER_NUM_EPOCHS = 2
TRANSFORMER_LEARNING_RATE = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01
TRANSFORMER_WARMUP_RATIO = 0.1

BERT_BASE_MODEL_NAME = "bert-base-uncased"
MENTAL_BERT_MODEL_NAME = "mental/mental-bert-base-uncased"

def set_global_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed()
print("✅ Config ready")

✅ Config ready


In [47]:
if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(f"Clean dataset not found: {CLEAN_DATA_PATH}")

df = pd.read_csv(CLEAN_DATA_PATH)

required_cols = [TEXT_COL, MASKED_COL, TARGET_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in clean dataset: {missing_cols}")

for col in required_cols:
    df[col] = df[col].astype(str).str.strip()

df = df[
    df[TEXT_COL].ne("") &
    df[MASKED_COL].ne("") &
    df[TARGET_COL].ne("")
].dropna().reset_index(drop=True)

if df.empty:
    raise ValueError("Dataset is empty after cleaning checks.")

print("✅ Clean dataset loaded")
print("Shape:", df.shape)
display(df.head())
print(df[TARGET_COL].value_counts())

✅ Clean dataset loaded
Shape: (11271, 3)


,body,body_masked,category
0,I often find myself checking on old friends on...,I often find myself checking on old friends on...,Depression
1,this is not a post asking if i have add depres...,this is not a post asking if i have add [CONDI...,ADHD
2,i went to my gp the other day to pick a new pr...,i went to my gp the other day to pick a new pr...,ADHD
3,im a morning person and always envied stories ...,im a morning person and always envied stories ...,ADHD
4,im preparing to be rediagnosed for adhd as im ...,im preparing to be rediagnosed for [CONDITION]...,ADHD


category
ADHD             2003
Anxiety          1918
Bipolar          1824
BPD              1618
Autism           1497
Depression       1433
schizophrenia     978
Name: count, dtype: int64


In [48]:
idx = np.arange(len(df))

train_idx, test_idx = train_test_split(
    idx,
    test_size=0.15,
    stratify=df[TARGET_COL],
    random_state=RANDOM_STATE
)

df_temp_train = df.loc[train_idx].reset_index(drop=True).copy()
df_test = df.loc[test_idx].reset_index(drop=True).copy()

temp_idx = np.arange(len(df_temp_train))

train_idx_2, val_idx = train_test_split(
    temp_idx,
    test_size=0.1765,  # ~15% of total when test is already 15%
    stratify=df_temp_train[TARGET_COL],
    random_state=RANDOM_STATE
)

df_train = df_temp_train.loc[train_idx_2].reset_index(drop=True).copy()
df_val = df_temp_train.loc[val_idx].reset_index(drop=True).copy()

print("Train size:", len(df_train))
print("Val size  :", len(df_val))
print("Test size :", len(df_test))

print("\nTrain distribution:")
print(df_train[TARGET_COL].value_counts(normalize=True).round(4))

print("\nVal distribution:")
print(df_val[TARGET_COL].value_counts(normalize=True).round(4))

print("\nTest distribution:")
print(df_test[TARGET_COL].value_counts(normalize=True).round(4))

Train size: 7889
Val size  : 1691
Test size : 1691

Train distribution:
category
ADHD             0.1778
Anxiety          0.1701
Bipolar          0.1619
BPD              0.1435
Autism           0.1328
Depression       0.1271
schizophrenia    0.0867
Name: proportion, dtype: float64

Val distribution:
category
ADHD             0.1774
Anxiety          0.1703
Bipolar          0.1614
BPD              0.1437
Autism           0.1331
Depression       0.1271
schizophrenia    0.0869
Name: proportion, dtype: float64

Test distribution:
category
ADHD             0.1774
Anxiety          0.1703
Bipolar          0.1620
BPD              0.1437
Autism           0.1325
Depression       0.1271
schizophrenia    0.0869
Name: proportion, dtype: float64


In [49]:
label_encoder = LabelEncoder()
label_encoder.fit(df_train[TARGET_COL])

df_train["label"] = label_encoder.transform(df_train[TARGET_COL])
df_val["label"] = label_encoder.transform(df_val[TARGET_COL])
df_test["label"] = label_encoder.transform(df_test[TARGET_COL])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}
num_labels = len(label_encoder.classes_)

print("✅ Labels encoded")
print("Classes:", list(label_encoder.classes_))
print("num_labels:", num_labels)

✅ Labels encoded
Classes: ['ADHD', 'Anxiety', 'Autism', 'BPD', 'Bipolar', 'Depression', 'schizophrenia']
num_labels: 7


In [50]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(df_train["label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df_train["label"]
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("✅ Class weights ready")
print(class_weights)

✅ Class weights ready
[0.80327869 0.83979136 1.07538168 0.99558304 0.8825372  1.12362911
 1.64766082]


In [51]:
@dataclass
class TransformerTextDataset(torch.utils.data.Dataset):
    texts: list
    labels: list
    tokenizer: object
    max_length: int

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        encoding["labels"] = int(self.labels[idx])
        return encoding

In [52]:
CRITICAL_LABELS = ["Bipolar", "schizophrenia"]

def critical_recall_score(y_true, y_pred, label_names):
    report = classification_report(
        y_true,
        y_pred,
        target_names=label_names,
        output_dict=True,
        zero_division=0
    )

    recalls = []
    for label in CRITICAL_LABELS:
        if label in report:
            recalls.append(report[label]["recall"])

    return float(np.mean(recalls)) if recalls else 0.0


def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    label_names = list(label_encoder.classes_)
    y_true_names = label_encoder.inverse_transform(labels)
    y_pred_names = label_encoder.inverse_transform(preds)

    f1_macro = f1_score(y_true_names, y_pred_names, average="macro", zero_division=0)
    recall_macro = recall_score(y_true_names, y_pred_names, average="macro", zero_division=0)
    critical_recall = critical_recall_score(y_true_names, y_pred_names, label_names)

    return {
        "f1_macro": float(f1_macro),
        "recall_macro": float(recall_macro),
        "critical_recall": float(critical_recall),
    }

In [53]:
if USE_MASKED_TEXT:
    X_train_texts = df_train[MASKED_COL].reset_index(drop=True)
    X_val_texts = df_val[MASKED_COL].reset_index(drop=True)
    X_test_texts = df_test[MASKED_COL].reset_index(drop=True)
else:
    X_train_texts = df_train[TEXT_COL].reset_index(drop=True)
    X_val_texts = df_val[TEXT_COL].reset_index(drop=True)
    X_test_texts = df_test[TEXT_COL].reset_index(drop=True)

X_test_raw_texts = df_test[TEXT_COL].reset_index(drop=True)

y_train_ids = df_train["label"].reset_index(drop=True).astype(int)
y_val_ids = df_val["label"].reset_index(drop=True).astype(int)
y_test_ids = df_test["label"].reset_index(drop=True).astype(int)

print("✅ Texts and labels prepared")
print("Train texts:", X_train_texts.shape)
print("Val texts  :", X_val_texts.shape)
print("Test texts :", X_test_texts.shape)

✅ Texts and labels prepared
Train texts: (7889,)
Val texts  : (1691,)
Test texts : (1691,)


In [54]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [64]:
def train_and_evaluate_transformer(model_name, run_name):
    output_dir = TRANSFORMER_REPORTS_DIR / run_name

    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_dataset = TransformerTextDataset(
        texts=X_train_texts.tolist(),
        labels=y_train_ids.tolist(),
        tokenizer=tokenizer,
        max_length=TRANSFORMER_MAX_LENGTH,
    )

    val_dataset = TransformerTextDataset(
        texts=X_val_texts.tolist(),
        labels=y_val_ids.tolist(),
        tokenizer=tokenizer,
        max_length=TRANSFORMER_MAX_LENGTH,
    )

    test_dataset = TransformerTextDataset(
        texts=X_test_texts.tolist(),
        labels=y_test_ids.tolist(),
        tokenizer=tokenizer,
        max_length=TRANSFORMER_MAX_LENGTH,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=1,
    )

    model.to(device)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_fn,
        data_collator=data_collator,
        class_weights=class_weights_tensor,
    )

    trainer.train()

    eval_metrics = trainer.evaluate(eval_dataset=test_dataset)

    predictions = trainer.predict(test_dataset)
    y_pred_ids = np.argmax(predictions.predictions, axis=1)
    y_true_ids = np.array(y_test_ids)

    y_pred_labels = label_encoder.inverse_transform(y_pred_ids)
    y_true_labels = label_encoder.inverse_transform(y_true_ids)

    f1_macro_value = f1_score(
        y_true_labels,
        y_pred_labels,
        average="macro",
        zero_division=0,
    )

    recall_macro_value = recall_score(
        y_true_labels,
        y_pred_labels,
        average="macro",
        zero_division=0,
    )

    critical_recall_value = critical_recall_score(
        y_true_labels,
        y_pred_labels,
        list(label_encoder.classes_),
    )

    accuracy_value = accuracy_score(y_true_labels, y_pred_labels)

    metrics = {
        "model_name": model_name,
        "run_name": run_name,
        "f1_macro": float(f1_macro_value),
        "recall_macro": float(recall_macro_value),
        "critical_recall": float(critical_recall_value),
        "accuracy": float(accuracy_value),
    }

    class_report = classification_report(
        y_true_labels,
        y_pred_labels,
        output_dict=True,
        zero_division=0,
    )

    class_recall_rows = []
    for class_name in label_encoder.classes_:
        if class_name in class_report:
            class_recall_rows.append({
                "model_name": model_name,
                "run_name": run_name,
                "label": class_name,
                "recall": float(class_report[class_name]["recall"]),
                "support": int(class_report[class_name]["support"]),
            })

    class_recall_df = pd.DataFrame(class_recall_rows)
    class_recall_path = output_dir / f"{run_name}_class_recall.csv"
    class_recall_df.to_csv(class_recall_path, index=False)

    results_df = pd.DataFrame([metrics])
    results_path = output_dir / f"{run_name}_metrics.csv"
    results_df.to_csv(results_path, index=False)

    predictions_df = pd.DataFrame({
        "text": X_test_texts.tolist(),
        "text_raw": X_test_raw_texts.tolist(),
        "y_true": y_true_labels,
        "y_pred": y_pred_labels,
    })
    predictions_path = output_dir / f"{run_name}_predictions.csv"
    predictions_df.to_csv(predictions_path, index=False)

    return {
        "metrics": metrics,
        "eval_metrics": eval_metrics,
        "results_df": results_df,
        "class_recall_df": class_recall_df,
        "predictions_df": predictions_df,
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
    }

In [65]:
# ============================================================
# TRANSFORMER GLOBAL CONFIG
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRANSFORMER_MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 0

print("✅ Transformer config ready")
print("device:", device)
print("TRANSFORMER_MAX_LENGTH:", TRANSFORMER_MAX_LENGTH)
print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)
print("EVAL_BATCH_SIZE:", EVAL_BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("LEARNING_RATE:", LEARNING_RATE)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("WARMUP_STEPS:", WARMUP_STEPS)

✅ Transformer config ready
device: cuda
TRANSFORMER_MAX_LENGTH: 256
TRAIN_BATCH_SIZE: 8
EVAL_BATCH_SIZE: 8
NUM_EPOCHS: 2
LEARNING_RATE: 2e-05
WEIGHT_DECAY: 0.01
WARMUP_STEPS: 0


In [66]:
# ============================================================
# TRANSFORMER METRICS
# ============================================================

def compute_metrics_transformers(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
    }

In [67]:
bert_base_results = train_and_evaluate_transformer(
    model_name=BERT_BASE_MODEL_NAME,
    run_name="bert_base_masked" if USE_MASKED_TEXT else "bert_base_raw"
)

pd.DataFrame([bert_base_results["metrics"]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Critical Recall
1,1.020176,0.715168,0.768902,0.773898,0.680534
2,0.493748,0.722780,0.780209,0.781324,0.707745


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

,model_name,run_name,f1_macro,recall_macro,critical_recall,accuracy
0,bert-base-uncased,bert_base_raw,0.793164,0.797068,0.749069,0.80071


In [68]:
mental_bert_results = train_and_evaluate_transformer(
    model_name=MENTAL_BERT_MODEL_NAME,
    run_name="mental_bert_masked" if USE_MASKED_TEXT else "mental_bert_raw"
)

pd.DataFrame([mental_bert_results["metrics"]])

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if 

Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Critical Recall
1,0.867791,0.696572,0.769905,0.775212,0.701204
2,0.441470,0.716576,0.785691,0.787716,0.723443


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

,model_name,run_name,f1_macro,recall_macro,critical_recall,accuracy
0,mental/mental-bert-base-uncased,mental_bert_raw,0.809424,0.812002,0.75612,0.816085


In [69]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("hungingfaces")

if hf_token:
    login(token=hf_token)
    print("✅ Hugging Face login successful")
else:
    print("⚠️ No Hugging Face token found in Colab secrets")

✅ Hugging Face login successful


In [ ]:
results_to_compare = []

if "bert_base_results" in globals() and bert_base_results is not None:
    results_to_compare.append(bert_base_results["metrics"])

if "mental_bert_results" in globals() and mental_bert_results is not None:
    results_to_compare.append(mental_bert_results["metrics"])

if not results_to_compare:
    raise ValueError("No transformer results available to compare.")

transformer_results_df = pd.DataFrame(results_to_compare)

print("colonnes disponibles:")
print(transformer_results_df.columns.tolist())

display(transformer_results_df)

Colunas disponíveis:
['model_name', 'run_name', 'f1_macro', 'recall_macro', 'critical_recall', 'accuracy']


,model_name,run_name,f1_macro,recall_macro,critical_recall,accuracy
0,bert-base-uncased,bert_base_raw,0.793164,0.797068,0.749069,0.800710
1,mental/mental-bert-base-uncased,mental_bert_raw,0.809424,0.812002,0.756120,0.816085


In [70]:
results_to_compare = []

if "bert_base_results" in globals() and bert_base_results is not None:
    results_to_compare.append(bert_base_results["metrics"])

if "mental_bert_results" in globals() and mental_bert_results is not None:
    results_to_compare.append(mental_bert_results["metrics"])

if not results_to_compare:
    raise ValueError("No transformer results available to compare.")

transformer_results_df = pd.DataFrame(results_to_compare).sort_values(
    by=["critical_recall", "f1_macro", "recall_macro"],
    ascending=[False, False, False]
).reset_index(drop=True)

transformer_results_df["family"] = "Transformer"

transformer_results_df.to_csv(TRANSFORMER_RESULTS_PATH, index=False)

print(f"✅ Transformer results saved to: {TRANSFORMER_RESULTS_PATH}")
display(transformer_results_df)

✅ Transformer results saved to: /content/mental_health_final_project/reports/transformers/transformer_test_results.csv


,model_name,run_name,f1_macro,recall_macro,critical_recall,accuracy,family
0,mental/mental-bert-base-uncased,mental_bert_raw,0.809424,0.812002,0.756120,0.816085,Transformer
1,bert-base-uncased,bert_base_raw,0.793164,0.797068,0.749069,0.800710,Transformer


In [72]:
class_recall_parts = []

if "bert_base_results" in globals() and bert_base_results is not None:
    class_recall_parts.append(bert_base_results["class_recall_df"])

if "mental_bert_results" in globals() and mental_bert_results is not None:
    class_recall_parts.append(mental_bert_results["class_recall_df"])

if not class_recall_parts:
    raise ValueError("No transformer class recall results available.")

transformer_class_recall_df = pd.concat(
    class_recall_parts,
    ignore_index=True
)

transformer_class_recall_df.to_csv(TRANSFORMER_CLASS_RECALL_PATH, index=False)

print(f"✅ Transformer class recall saved to: {TRANSFORMER_CLASS_RECALL_PATH}")
display(
    transformer_class_recall_df.sort_values(
        ["run_name", "recall"],
        ascending=[True, False]
    )
)

✅ Transformer class recall saved to: /content/mental_health_final_project/reports/transformers/transformer_class_recall.csv


,model_name,run_name,label,recall,support
0,bert-base-uncased,bert_base_raw,ADHD,0.886667,300
3,bert-base-uncased,bert_base_raw,BPD,0.872428,243
2,bert-base-uncased,bert_base_raw,Autism,0.825893,224
1,bert-base-uncased,bert_base_raw,Anxiety,0.784722,288
6,bert-base-uncased,bert_base_raw,schizophrenia,0.775510,147
4,bert-base-uncased,bert_base_raw,Bipolar,0.722628,274
5,bert-base-uncased,bert_base_raw,Depression,0.711628,215
7,mental/mental-bert-base-uncased,mental_bert_raw,ADHD,0.893333,300
10,mental/mental-bert-base-uncased,mental_bert_raw,BPD,0.872428,243
9,mental/mental-bert-base-uncased,mental_bert_raw,Autism,0.848214,224


In [73]:
prediction_parts = []

if "bert_base_results" in globals() and bert_base_results is not None:
    prediction_parts.append(
        bert_base_results["predictions_df"].assign(
            run_name=bert_base_results["metrics"]["run_name"]
        )
    )

if "mental_bert_results" in globals() and mental_bert_results is not None:
    prediction_parts.append(
        mental_bert_results["predictions_df"].assign(
            run_name=mental_bert_results["metrics"]["run_name"]
        )
    )

if not prediction_parts:
    raise ValueError("No transformer prediction results available.")

all_transformer_predictions = pd.concat(
    prediction_parts,
    ignore_index=True
)

all_transformer_predictions.to_csv(TRANSFORMER_PREDICTIONS_PATH, index=False)

print(f"✅ Transformer predictions saved to: {TRANSFORMER_PREDICTIONS_PATH}")
display(all_transformer_predictions.head())

✅ Transformer predictions saved to: /content/mental_health_final_project/reports/transformers/transformer_test_predictions.csv


,text,text_raw,y_true,y_pred,run_name
0,For the last 5 or 6 years I have lived about a...,For the last 5 or 6 years I have lived about a...,Anxiety,Anxiety,bert_base_raw
1,Skating was the biggest most persistent intere...,Skating was the biggest most persistent intere...,Autism,Autism,bert_base_raw
2,Am I turning into a woman\n\nI am crying often...,Am I turning into a woman\n\nI am crying often...,Depression,Depression,bert_base_raw
3,I have an interview oncampus placement drive t...,I have an interview oncampus placement drive t...,Depression,Anxiety,bert_base_raw
4,Help me\n\nI want to die i would be better off...,Help me\n\nI want to die i would be better off...,Depression,Depression,bert_base_raw


In [77]:
import os
print(os.listdir("/content"))

['.config', 'final_test_metrics.csv', 'mental_health_final_project', 'sample_data']


In [84]:
from pathlib import Path

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPORTS_DIR:", REPORTS_DIR)
print("CLASSICAL dir exists:", (REPORTS_DIR / "classical").exists())
print("TABLES/classical dir exists:", (REPORTS_DIR / "tables" / "classical").exists())

if (REPORTS_DIR / "classical").exists():
    print("\nFiles in reports/classical:")
    for p in (REPORTS_DIR / "classical").glob("*"):
        print(" -", p.name)

if (REPORTS_DIR / "tables" / "classical").exists():
    print("\nFiles in reports/tables/classical:")
    for p in (REPORTS_DIR / "tables" / "classical").glob("*"):
        print(" -", p.name)

PROJECT_ROOT: /content/mental_health_final_project
REPORTS_DIR: /content/mental_health_final_project/reports
CLASSICAL dir exists: False
TABLES/classical dir exists: False


In [86]:
from google.colab import files
uploaded = files.upload()

Saving final_test_metrics.csv to final_test_metrics (2).csv


In [89]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    COMPARISON_REPORTS_DIR.parent / "classical" / "final_test_metrics.csv",
    COMPARISON_REPORTS_DIR.parent / "classical" / "final_test_results.csv",
    COMPARISON_REPORTS_DIR.parent / "tables" / "classical" / "final_test_metrics.csv",
    COMPARISON_REPORTS_DIR.parent / "tables" / "classical" / "final_test_results.csv",
    Path("/content/final_test_metrics (2).csv"),
]

CLASSICAL_FINAL_RESULTS_PATH = None
classical_results_df = None

print("🔍 Searching for classical final results...")
for path in candidate_paths:
    print(f" - {path} | exists={path.exists()}")
    if path.exists():
        CLASSICAL_FINAL_RESULTS_PATH = path
        classical_results_df = pd.read_csv(path).copy()
        print(f"\n✅ Loaded classical results from: {path}")
        break

if classical_results_df is None:
    raise FileNotFoundError(
        "Classical final results not found in any candidate path.\n"
        + "\n".join(str(p) for p in candidate_paths)
    )

print("\nColumns found:")
print(list(classical_results_df.columns))

required_classical_cols = {
    "champion_model",
    "text_variant",
    "f1_macro",
    "recall_macro",
    "critical_recall",
}
missing_classical_cols = required_classical_cols - set(classical_results_df.columns)

if missing_classical_cols:
    raise ValueError(
        f"Missing columns in classical results: {missing_classical_cols}\n"
        f"Available columns: {list(classical_results_df.columns)}"
    )

classical_results_df["model_name"] = classical_results_df["champion_model"]
classical_results_df["run_name"] = (
    classical_results_df["model_name"].astype(str)
    + " | classical | "
    + classical_results_df["text_variant"].astype(str)
)
classical_results_df["family"] = "Classical ML"

transformer_compare_df = (
    transformer_results_df.copy()
    if "transformer_results_df" in globals()
    else pd.DataFrame()
)

if not transformer_compare_df.empty and "family" not in transformer_compare_df.columns:
    transformer_compare_df["family"] = "Transformer"

comparison_parts = [
    classical_results_df[
        ["run_name", "model_name", "f1_macro", "recall_macro", "critical_recall", "family"]
    ]
]

if not transformer_compare_df.empty:
    comparison_parts.append(
        transformer_compare_df[
            ["run_name", "model_name", "f1_macro", "recall_macro", "critical_recall", "family"]
        ]
    )

comparison_df = pd.concat(
    comparison_parts,
    ignore_index=True
).sort_values(
    by=["critical_recall", "f1_macro", "recall_macro"],
    ascending=[False, False, False]
).reset_index(drop=True)

comparison_output_path = COMPARISON_REPORTS_DIR / "classical_vs_transformers.csv"
comparison_df.to_csv(comparison_output_path, index=False)

print(f"\n✅ Comparison saved to: {comparison_output_path}")
display(comparison_df)

🔍 Searching for classical final results...
 - /content/mental_health_final_project/reports/classical/final_test_metrics.csv | exists=False
 - /content/mental_health_final_project/reports/classical/final_test_results.csv | exists=False
 - /content/mental_health_final_project/reports/tables/classical/final_test_metrics.csv | exists=False
 - /content/mental_health_final_project/reports/tables/classical/final_test_results.csv | exists=False
 - /content/final_test_metrics (2).csv | exists=True

✅ Loaded classical results from: /content/final_test_metrics (2).csv

Columns found:
['champion_model', 'text_variant', 'f1_macro', 'recall_macro', 'critical_recall']

✅ Comparison saved to: /content/mental_health_final_project/reports/comparisons/classical_vs_transformers.csv


,run_name,model_name,f1_macro,recall_macro,critical_recall,family
0,mental_bert_raw,mental/mental-bert-base-uncased,0.809424,0.812002,0.756120,Transformer
1,bert_base_raw,bert-base-uncased,0.793164,0.797068,0.749069,Transformer
2,LinearSVC_balanced | classical | raw,LinearSVC_balanced,0.778927,0.778686,0.698068,Classical ML
